# 05 · Layer 2：Memory 與 Artifact

上一章的 Session 解決了「這段對話」的記憶。但還有兩個缺口：

| 需求 | 工具 | 類比 |
|---|---|---|
| 「上禮拜他跟我說過什麼？」 | **Memory** | 硬碟 |
| 「把這份 PDF 存起來，等下另一個 agent 要用」 | **Artifact** | 檔案系統 |

三者的分工：

```
Session   短期、結構化、這段對話    → state / events
Memory    長期、可搜尋、跨對話      → 過去的對話內容
Artifact  二進位／大檔案、有版本    → PDF、圖片、報告
```

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. Memory：把對話歸檔，之後可以搜尋

流程有三步：

1. 對話結束後，把整個 session **歸檔**進 memory
2. 新對話裡，agent 用 `load_memory` 工具**搜尋**過往記憶
3. Runner 建立時要把 `memory_service` 傳進去，否則工具會找不到後端

In [2]:
from google.adk.agents import LlmAgent
from google.adk.memory import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import load_memory

APP = "concept_track"
USER = "student"

session_service = InMemorySessionService()
memory_service = InMemoryMemoryService()

# 第一段對話：單純聊天，沒有 memory 工具
chatter = LlmAgent(
    name="chatter",
    model=get_model(),
    instruction="你是助理，用繁體中文簡短回答，並記下使用者說的事實。",
)
r1 = Runner(agent=chatter, app_name=APP, session_service=session_service)
sid1 = await new_session(r1)

print(await ask(r1, "我叫 Sean，養了一隻叫做 Mochi 的柴犬，牠今年三歲。", session_id=sid1))
print(await ask(r1, "我最近在學 Google ADK。", session_id=sid1))

你好，Sean！記住了，你養了一隻三歲的柴犬 Mochi。


好的，Sean，我記下你在學 Google ADK 了！


In [3]:
# 步驟 1：歸檔
past = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid1)
await memory_service.add_session_to_memory(past)
print(f"已把 {len(past.events)} 筆事件歸檔進 memory")

已把 4 筆事件歸檔進 memory


### 開一段全新的對話，看它記不記得

In [4]:
# 對照組：沒有 memory 工具的 agent
r_no_mem = Runner(agent=chatter, app_name=APP, session_service=session_service)
sid_no = await new_session(r_no_mem)
print("【沒有 memory】", await ask(r_no_mem, "我的狗叫什麼名字？", session_id=sid_no))

【沒有 memory】 您還沒有告訴我您的狗叫什麼名字。


In [5]:
# 實驗組：有 load_memory 工具，而且 Runner 掛了 memory_service
memory_agent = LlmAgent(
    name="memory_agent",
    model=get_model(),
    instruction=(
        "你可以存取使用者過去的對話紀錄。"
        "被問到跟過往有關的問題時，**先呼叫 load_memory 工具**搜尋，再用繁體中文回答。"
    ),
    tools=[load_memory],
)

r2 = Runner(
    agent=memory_agent,
    app_name=APP,
    session_service=session_service,
    memory_service=memory_service,  # ← 少了這行，load_memory 會失敗
)
sid2 = await new_session(r2)

print("【有 memory】", await ask(r2, "我的狗叫什麼名字？幾歲？", session_id=sid2, trace=True))

  🔧 [memory_agent] 呼叫 load_memory({'query': '狗 名字 幾歲'})
  ↩️  [memory_agent] load_memory 回傳 {'result': LoadMemoryResponse(memories=[])}


  💬 [memory_agent] 我目前在記憶中找不到關於你的狗的名字和年齡的紀錄。請問牠叫什麼名字、幾歲呢？告訴我之後我會記下來的！
【有 memory】 我目前在記憶中找不到關於你的狗的名字和年齡的紀錄。請問牠叫什麼名字、幾歲呢？告訴我之後我會記下來的！


## 1.5 📌 等一下——它查不到？

如果上面那格回你「找不到相關資訊」，**那不是你設定錯了**。
看一下 trace：`load_memory` 確實被呼叫了，但回傳 `memories=[]`。

原因藏在 `InMemoryMemoryService` 的實作裡。它的 docstring 寫得很白：

In [6]:
from google.adk.memory.in_memory_memory_service import InMemoryMemoryService as IMMS

print(IMMS.__doc__)

An in-memory memory service for prototyping purpose only.

Uses keyword matching instead of semantic search. A search returns at most
ten memories, the ones sharing the most words with the query.

This class is thread-safe, however, it should be used for testing and
development only.



**「Uses keyword matching instead of semantic search」**——它不是語意搜尋，
是**單純的字詞重疊比對**。而它斷詞用的是 `re.findall(r"\w+", text)`。

對英文沒問題，對中文是災難：

In [7]:
from google.adk.memory.in_memory_memory_service import _extract_words_lower

samples = [
    ("存進去的內容（中文）", "我養了一隻叫做 Mochi 的柴犬，牠今年三歲"),
    ("查詢的關鍵字（中文）", "狗 名字 幾歲"),
    ("存進去的內容（英文）", "I have a Shiba Inu named Mochi, he is three years old"),
    ("查詢的關鍵字（英文）", "dog name age Mochi"),
]
tokens = {}
for label, text in samples:
    tokens[label] = _extract_words_lower(text)
    print(f"{label}\n  {text}\n  → {sorted(tokens[label])}\n")

zh = tokens["存進去的內容（中文）"] & tokens["查詢的關鍵字（中文）"]
en = tokens["存進去的內容（英文）"] & tokens["查詢的關鍵字（英文）"]
print(f"中文的交集: {zh or '（空的 → 永遠查不到）'}")
print(f"英文的交集: {en}")

存進去的內容（中文）
  我養了一隻叫做 Mochi 的柴犬，牠今年三歲
  → ['mochi', '我養了一隻叫做', '牠今年三歲', '的柴犬']

查詢的關鍵字（中文）
  狗 名字 幾歲
  → ['名字', '幾歲', '狗']

存進去的內容（英文）
  I have a Shiba Inu named Mochi, he is three years old
  → ['a', 'have', 'he', 'i', 'inu', 'is', 'mochi', 'named', 'old', 'shiba', 'three', 'years']

查詢的關鍵字（英文）
  dog name age Mochi
  → ['age', 'dog', 'mochi', 'name']

中文的交集: （空的 → 永遠查不到）
英文的交集: {'mochi'}


中文沒有空白分隔，`\w+` 會把一整串黏成一個 token
（`我養了一隻叫做`、`牠今年三歲`），跟查詢的 `狗`、`名字`、`幾歲` 永遠對不上。

**唯一會命中的是 `mochi`**——因為它是拉丁字母。驗證一下：

In [8]:
result = await memory_service.search_memory(
    app_name=APP, user_id=USER, query="Mochi"
)
print(f"用 'Mochi' 查 → {len(result.memories)} 筆")
for m in result.memories[:3]:
    text = "".join(p.text or "" for p in (m.content.parts or []))
    print(f"  ▪ {text[:60]}")

result_zh = await memory_service.search_memory(
    app_name=APP, user_id=USER, query="狗 名字 幾歲"
)
print(f"\n用 '狗 名字 幾歲' 查 → {len(result_zh.memories)} 筆")

用 'Mochi' 查 → 2 筆
  ▪ 我叫 Sean，養了一隻叫做 Mochi 的柴犬，牠今年三歲。
  ▪ 你好，Sean！記住了，你養了一隻三歲的柴犬 Mochi。

用 '狗 名字 幾歲' 查 → 0 筆


### 這件事的實際教訓

| | |
|---|---|
| `InMemoryMemoryService` 的定位 | 官方寫明 **for prototyping purpose only** |
| 它的搜尋方式 | 字詞重疊，**不是語意搜尋** |
| 中文的下場 | 斷詞失效，實質上查不到東西 |
| 正式環境要換成 | `VertexAiMemoryBankService` 或 `VertexAiRagMemoryService`（真正的語意檢索） |

換句話說：**這一節示範的是機制，不是可用的中文記憶系統**。
機制（歸檔 → 搜尋 → 注入）是對的，換掉後端就能用。

這也是為什麼你不該用 `InMemory*` 系列的表現去評估「ADK 的 memory 好不好用」。

### Memory 不是自動的

除了上面那個坑，還有三件事很容易被忽略：

1. **歸檔要你自己呼叫** `add_session_to_memory()`。ADK 不會自動幫你存。
   實務上通常掛在對話結束、或每 N 輪觸發。
2. **搜尋也要模型主動呼叫工具**。模型覺得不需要就不會查。
3. **`memory_service` 一定要傳給 Runner**，否則 `load_memory` 拿不到後端。

In [9]:
# 示範 3：忘了傳 memory_service 會怎樣
r_broken = Runner(agent=memory_agent, app_name=APP, session_service=session_service)
sid_broken = await new_session(r_broken)
try:
    print(await ask(r_broken, "我的狗叫什麼？", session_id=sid_broken))
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:200]}")

ValueError: Memory service is not available.


### 另一個選擇：`preload_memory`

`load_memory` 是「模型自己決定要不要查」。如果你希望**每一輪都自動把相關記憶
塞進上下文**，用 `preload_memory` —— 它不需要模型主動呼叫。

代價是每輪都多一段 context（更貴），好處是不會漏查。

In [10]:
from google.adk.tools import preload_memory

auto_memory_agent = LlmAgent(
    name="auto_memory_agent",
    model=get_model(),
    instruction="你是助理，用繁體中文簡短回答。",
    tools=[preload_memory],
)
r3 = Runner(
    agent=auto_memory_agent,
    app_name=APP,
    session_service=session_service,
    memory_service=memory_service,
)
sid3 = await new_session(r3)
print(await ask(r3, "我最近在學什麼？", session_id=sid3, trace=True))

  💬 [auto_memory_agent] 我目前沒有記錄您最近在學習什麼。如果您告訴我，我可以幫您記住！
我目前沒有記錄您最近在學習什麼。如果您告訴我，我可以幫您記住！


注意 `preload_memory` 這裡**沒有出現工具呼叫**——它是在送出請求前就把記憶
塞進上下文，不需要模型決定。這正是它跟 `load_memory` 的差別。

（至於內容有沒有被找到，一樣受制於上面說的中文斷詞問題。
機制對了，但 `InMemoryMemoryService` 的檢索能力就是這樣。）

## 2. Artifact：agent 系統的檔案系統

Memory 存的是「對話」，Artifact 存的是「檔案」——報告、PDF、圖片、音訊。
內容用 `types.Part` 表示，所以文字和二進位都能放。

兩個關鍵：

- **版本**：同名存第二次會產生 version 1，不會覆蓋掉 version 0
- **作用域**：檔名加 `user:` 前綴 → 跨 session；不加 → 只有這個 session 看得到

In [11]:
from google.adk.artifacts import InMemoryArtifactService
from google.adk.tools import ToolContext
from google.genai import types


async def save_report(topic: str, content: str, tool_context: ToolContext) -> dict:
    """把一份 Markdown 報告存成 artifact。

    Args:
        topic: 報告標題。
        content: 報告內文（Markdown）。
    """
    md = f"# {topic}\n\n{content}\n"
    version = await tool_context.save_artifact(
        filename="user:report.md",  # user: 前綴 → 跨 session 都讀得到
        artifact=types.Part(text=md),
    )
    return {"saved": True, "version": version, "chars": len(md)}


async def read_report(tool_context: ToolContext) -> dict:
    """讀出最新版本的報告。"""
    part = await tool_context.load_artifact(filename="user:report.md")
    if part is None:
        return {"found": False}
    return {"found": True, "content": part.text}


writer = LlmAgent(
    name="writer",
    model=get_model(),
    instruction="使用者要你寫東西時，呼叫 save_report 存起來，再用一句話回報。",
    tools=[save_report],
)

reader = LlmAgent(
    name="reader",
    model=get_model(),
    instruction="使用者問上次寫的內容時，呼叫 read_report 取出，並原樣呈現。",
    tools=[read_report],
)

In [12]:
artifact_service = InMemoryArtifactService()

# Session A：寫
rw = Runner(
    agent=writer,
    app_name=APP,
    session_service=session_service,
    artifact_service=artifact_service,
)
sid_a = await new_session(rw)
print(await ask(rw, "幫我寫一份『什麼是 AI Agent』的兩句話摘要並存檔。",
                session_id=sid_a, trace=True))

  🔧 [writer] 呼叫 save_report({'topic': '什麼是 AI Agent', 'content': 'AI Agent 是一種能夠自主感知環境、進行推理並採取行動以達成特定目標的人工智慧系統。它不僅能回答問題，還能串接工具、規劃步驟並獨立完成複雜的任務。'})
  ↩️  [writer] save_report 回傳 {'saved': True, 'version': 0, 'chars': 93}


  💬 [writer] 「什麼是 AI Agent」的兩句話摘要已經成功存檔囉！
「什麼是 AI Agent」的兩句話摘要已經成功存檔囉！


In [13]:
# Session B：完全不同的對話，照樣讀得到
rr = Runner(
    agent=reader,
    app_name=APP,
    session_service=session_service,
    artifact_service=artifact_service,  # 共用同一個 artifact service
)
sid_b = await new_session(rr)
print(await ask(rr, "我上次存的報告內容是什麼？", session_id=sid_b))

# 什麼是 AI Agent

AI Agent 是一種能夠自主感知環境、進行推理並採取行動以達成特定目標的人工智慧系統。它不僅能回答問題，還能串接工具、規劃步驟並獨立完成複雜的任務。


### 版本控制

同一個檔名再存一次，會得到新的 version，舊的仍然在。

In [14]:
versions = await artifact_service.list_versions(
    app_name=APP, user_id=USER, session_id=sid_a, filename="user:report.md"
)
print("目前版本:", versions)

# 再存一次
await ask(rw, "改寫成一句話就好，重新存檔。", session_id=sid_a)

versions = await artifact_service.list_versions(
    app_name=APP, user_id=USER, session_id=sid_a, filename="user:report.md"
)
print("再存一次之後的版本:", versions)

for v in versions:
    part = await artifact_service.load_artifact(
        app_name=APP, user_id=USER, session_id=sid_a,
        filename="user:report.md", version=v,
    )
    print(f"\n--- version {v} ---")
    print((part.text or "")[:180])

目前版本: [0]


再存一次之後的版本: [0, 1]

--- version 0 ---
# 什麼是 AI Agent

AI Agent 是一種能夠自主感知環境、進行推理並採取行動以達成特定目標的人工智慧系統。它不僅能回答問題，還能串接工具、規劃步驟並獨立完成複雜的任務。


--- version 1 ---
# 什麼是 AI Agent

AI Agent 是一種能夠自主感知、規劃並串接工具以獨立完成複雜目標的智慧系統。



### `user:` 前綴到底差在哪

把前綴拿掉，同樣的程式在 session B 就讀不到了：

In [15]:
await artifact_service.save_artifact(
    app_name=APP, user_id=USER, session_id=sid_a,
    filename="session_only.md", artifact=types.Part(text="只有 session A 看得到"),
)

for label, sid in (("session A（存檔的那個）", sid_a), ("session B（另一個）", sid_b)):
    got = await artifact_service.load_artifact(
        app_name=APP, user_id=USER, session_id=sid, filename="session_only.md"
    )
    print(f"{label:24s} → {'讀得到' if got else '讀不到'}")

session A（存檔的那個）         → 讀得到
session B（另一個）           → 讀不到


## 3. 上正式環境時換什麼

三個服務都是介面，換實作不用改 agent：

| 開發用 | 正式環境 |
|---|---|
| `InMemorySessionService` | `DatabaseSessionService` / `VertexAiSessionService` |
| `InMemoryMemoryService` | `VertexAiMemoryBankService` / `VertexAiRagMemoryService` |
| `InMemoryArtifactService` | `GcsArtifactService` |

In [16]:
import google.adk.memory as mem
import google.adk.artifacts as art

print("memory 可用實作 :", [n for n in dir(mem) if n.endswith("MemoryService")])
print("artifact 可用實作:", [n for n in dir(art) if n.endswith("ArtifactService")])

memory 可用實作 : ['BaseMemoryService']
artifact 可用實作: ['BaseArtifactService']


## 本章重點

- **Session / Memory / Artifact 是三件不同的事**：這段對話 / 過去的對話 / 檔案。
- **⚠️ `InMemoryMemoryService` 是字詞重疊比對、不是語意搜尋，而且對中文幾乎失效**
  （`\w+` 斷不開中文）。官方定位就是 prototyping only，正式環境要換
  `VertexAiMemoryBankService` / `VertexAiRagMemoryService`。
- **Memory 不是自動的**：歸檔要自己呼叫 `add_session_to_memory()`，
  搜尋要模型主動呼叫 `load_memory`，而且 `memory_service` 一定要傳給 Runner。
- **`preload_memory`** 是「每輪自動塞」的版本，不會漏查但比較貴。
- **Artifact 有版本**，同名再存不會蓋掉舊的。
- **`user:` 前綴讓 artifact 跨 session**，不加前綴就只有該 session 看得到。
- 換正式環境只要換服務實作，agent 程式碼不動。

## 動手練習

1. 把第 1 節存進 memory 的內容改成英文，重跑 `load_memory`，
   確認英文的關鍵字比對是有效的。
2. 讓 `save_report` 改存成 session-scoped（拿掉 `user:`），
   重跑 session B，確認讀不到。
3. 寫一個工具 `list_my_files(tool_context)`，用
   `tool_context.list_artifacts()` 列出目前所有檔案，掛給 `reader` 用。

---
**下一站 → `06_app_callbacks_plugins.ipynb`**：
App 容器、六個 callback 掛載點，以及 Plugin 為什麼會蓋掉你的 callback。